Обучение моделей определению тональности (положительный, нейтральный, отрицательный) комментариев

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
# from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV

# from catboost import CatBoostClassifier
# from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier

### Классы:

0 - нейтральный

1 - позитивный 

2 - негативный

In [2]:
dataset = pd.read_csv("../src/data/data.csv")

In [ ]:
dataset.columns

In [ ]:
dataset.shape

In [9]:
dataset["sentiments"].value_counts()

sentiments
1    9503
0    6300
2    1534
Name: count, dtype: int64

In [ ]:
data_neutral= dataset[dataset["sentiments"] == 0]
data_pos = dataset[dataset["sentiments"] == 1]
data_negative = dataset[dataset["sentiments"] == 2]

In [18]:
def metrics(val, pred, title):
    print(f"{title}:")
    print(f"Точность: {accuracy_score(val, pred)}")
    print(f"f1: {f1_score(val, pred, average='macro')}")
    print(f"f1_weighted {f1_score(val, pred, average='weighted')}")
    print(f"precision_macro {precision_score(val, pred, average='macro')}")
    print(f"recall_macro {recall_score(val, pred, average='macro')}")


In [ ]:
def bar_plot(values):
    plt.figure(figsize=(10, 6))
    plt.grid(True)
    plt.title("Соотношение положительных и отрицательных отзывов")
    names = ["neutral", "positive", "negative"]

    plt.bar(names, values)
    plt.show()

In [ ]:
bar_plot([data_negative .shape[0], data_pos.shape[0], data_neutral.shape[0]])

### Разделение данных

In [3]:
X, y = dataset["cleaned_review"], dataset["sentiments"]

In [4]:
X_train_temp, X_test, y_train_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_temp, y_train_temp, test_size=0.2, stratify=y_train_temp)

### Базовая модель LinearSVC

In [ ]:
base_pipeline = Pipeline([
    ("tfid", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english", min_df = 2)),
    ("cclf", CalibratedClassifierCV(estimator = LinearSVC(C=1.0, max_iter=10000,  class_weight='balanced', loss = 'squared_hinge'), cv = 5))
])

base_pipeline.fit(X_train, y_train)

baseline_predict = base_pipeline.predict(X_val)
baseline_predict_proba = base_pipeline.predict_proba(X_val)

In [25]:
def predict_model(model_pipeline, text):
    proba = model_pipeline.predict([text])
    return proba

In [26]:
metrics(y_val, baseline_predict, "LinearSVC")

LinearSVC:
Точность: 0.773972602739726
f1: 0.6224475238375118
f1_weighted 0.7578039796165866
precision_macro 0.6964199769554403
recall_macro 0.6062840191961071


Улучшение базовой модели с помощью RandomizerSearchCV 

In [ ]:
baseline_dict = {
    "tfid__max_features": [5000, 10000, 15000],
    "tfid__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfid__min_df": [1, 2, 3],
    "svc__C": [0.1, 1.0, 10.0],
    "svc__loss": ["hinge", "squared_hinge"]
}

In [ ]:
random_search = RandomizedSearchCV(
    base_pipeline,
    param_distributions = baseline_dict,
    n_iter = 30,
    cv = 3,
    scoring = "f1",
    n_jobs = -1
)

In [ ]:
random_search.fit(X_train, y_train)
print(f"Лучшие параметры: {random_search.best_params_}")

### Логистическая регрессия

In [ ]:
pipe = Pipeline([
    ("tfid", TfidfVectorizer(max_features=10000, stop_words="english")),
    ("lt", LogisticRegression(max_iter = 2000, max_features = 10000, class_weight = "balanced"))
])


pipe.fit(X_train, y_train)
ls_preds = pipe.predict(X_val)

In [28]:
metrics(y_val, ls_preds, "Логистическая регрессия")

Логистическая регрессия:
Точность: 0.7379235760634463
f1: 0.6613861890649037
f1_weighted 0.7513109281321046
precision_macro 0.6588905159863158
recall_macro 0.7206182045467759


### Catboost

In [ ]:
df_train = pd.DataFrame({'text': X_train, 'label': y_train})
df_val = pd.DataFrame({'text': X_val, 'label': y_val})

In [ ]:
cat_boost = CatBoostClassifier(
    iterations = 500,
    depth = 5,
    learning_rate = 0.01,
    loss_function = "MultiClass",
    early_stopping_rounds = 50
)

cat_boost.fit(    
    df_train[["text"]], 
    y = df_train["label"],
    text_features=['text'], 
    eval_set=[(df_val[['text']], df_val['label'])],
    use_best_model = True
)

cat_predict = cat_boost.predict(df_val[["text"]])

In [ ]:
metrics(df_val['label'], cat_predict, "CatBoost")

###  Multinomial Naive Bayes

In [ ]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000, 
        ngram_range=(1,2), 
        stop_words="english")),

    ("nb", MultinomialNB(
        alpha=0.01, 
        class_prior=None, 
        fit_prior=True))
])

nb_pipeline.fit(X_train, y_train)

nb_predict = nb_pipeline.predict(X_val)

In [ ]:
np_param_grid = {
    "nb__alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
    "tfidf__ngram_range": [(1,2), (1,2), (2,2), (1,3)]
}

In [ ]:
nb_search = GridSearchCV(
    nb_pipeline,
    np_param_grid,
    cv = 3, 
    scoring = "f1",
    n_jobs = -1)

nb_search.fit(X_train, y_train)

In [ ]:
nb_search.best_params_

In [ ]:
metrics(y_val, nb_predict, "MultinomialNB")

Xgboost

In [6]:
xg = XGBClassifier(
    n_estimators = 100,
    max_depth = 5,
    learning_rate = 0.1,
    objective='multi:softprob',
    num_class = 3,
    random_state = 42,
    eval_metric='mlogloss'
)

In [ ]:
xg_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
        max_features=10000, 
        ngram_range=(1,2), 
        stop_words="english")),
        ("classifier",
       XGBClassifier(
    n_estimators = 100,
    max_depth = 5,
    learning_rate = 0.1,
    objective='multi:softprob',
    num_class = 3,
    random_state = 42,
    eval_metric='mlogloss'
))
])

In [8]:
xg.fit(X_train, y_train)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:cleaned_review: object